# 06 · Sensibilidad de α y β de la Weighted Focal Loss
Responde a: *¿por qué esos α y β? ¿fueron elegidos arbitrariamente?* β se amplía más allá de 100 (antes quedó en el borde del grid). Selección por **HSS** en Validación (no solo RMSE en eventos, que premia sobreestimar).

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from s4.config import cargar_config, variante, ruta
from s4.pipeline import preparar_datos, correr_variante
from s4.registry import leer, resumir_semillas
from s4.train import configurar_gpu
from s4 import features as ft
configurar_gpu()
cfg = cargar_config('../config/ablacion_2025_2026.yaml')
datos = preparar_datos(cfg)
CONJUNTO = "E_completo"      # <-- el mejor de la ablación

In [ ]:
for beta in [10, 25, 50, 100, 150, 200, 300]:
    for alpha in [1.0, 1.5, 2.0]:
        c = variante(cfg, **{"perdida.beta": float(beta), "perdida.alpha": alpha})
        correr_variante(c, datos, ft.CONJUNTOS[CONJUNTO], f"a{alpha}_b{beta}", "alpha_beta", semillas=[42])

In [ ]:
reg = leer(ruta(cfg, "experimentos") / f"registro_{cfg['nombre']}.csv", grupo="alpha_beta", particion="val")
reg[["alpha","beta"]] = reg["modelo"].str.extract(r"a([\d.]+)_b([\d.]+)").astype(float)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, met in zip(axes, ["HSS", "RMSE_evento", "RMSE_global"]):
    for a, g in reg.groupby("alpha"):
        ax.plot(g["beta"], g[met], "o-", label=f"α={a}")
    ax.set_xscale("log"); ax.set_xlabel("β"); ax.set_title(met); ax.legend()
plt.tight_layout(); plt.savefig(ruta(cfg,"reportes")/"figures/fig_sensibilidad_alpha_beta.png", dpi=200); plt.show()
reg.pivot_table(index="beta", columns="alpha", values="HSS").round(3)

Confirmar los 2–3 mejores pares (α, β) con las 3 semillas (`semillas=None`) antes de fijarlos en el YAML.